# Воспроизведение выбранного решения

Этот ноутбук восстанавливает замороженные best checkpoints, порядок классов, preprocessing и TTA из manifest, выбранного на val. Без повторного обучения и подбора параметров по test. Сохранённые веса обязательны; их SHA-256 проверяется. Текущий manifest выбран после завершённых ResNet101 TTA/crop сравнений. Новый Average+Max кандидат ещё не обучен и не участвует в выборе.


In [ ]:
from pathlib import Path
import json
import sys
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src/inference.py").exists():
    raise RuntimeError("Откройте ноутбук из корня проекта или notebooks/")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.inference import run_selected_inference


In [ ]:
SELECTION_MANIFEST = "artifacts/logs/resnet101_global_local_001/final_selection.json"
RAW_DIR = PROJECT_ROOT / "data/raw"
DEVICE = "auto"
BATCH_SIZE = 16
PREDICT_TEST = False  # Включите только для выбранного финального решения
RUN_ID = "final_selected_001"  # Новый уникальный ID

selection = json.loads((PROJECT_ROOT / SELECTION_MANIFEST).read_text(encoding="utf-8"))
print("Рецепт:", selection["recipe"], "| val Accuracy:", selection["val_accuracy"])
display(pd.DataFrame(selection["members"])[["name", "tta", "weight", "checkpoint"]])


In [ ]:
if PREDICT_TEST:
    result = run_selected_inference(selection, RUN_ID, split="test", project_root=PROJECT_ROOT,
                                    raw_dir=RAW_DIR, device=DEVICE, batch_size=BATCH_SIZE)
    print("Submission:", result["paths"]["submission"])
else:
    print("Test inference выключен. Для проверки replay на val используйте отдельный RUN_ID:")
    print('run_selected_inference(selection, "final_val_replay_001", split="val", project_root=PROJECT_ROOT)')


Для следующего fine-tuning используйте 03_pretrained.ipynb; для выбора ансамбля — 06_tta_ensembles.ipynb с новым inference ID. После выбора замените SELECTION_MANIFEST. Все пути в manifest относительно проекта; код не скачивает pretrained weights при восстановлении checkpoint. Submission строится в порядке sample_submission_example.csv; labels соответствуют сохранённому порядку классов.
